# 🧠 EEG Emotional Memory — GPU Ultra Pipeline v5.0

## Architecture
```
DATA (D:\eeg_competition)
  └─ 14 train subjects + 3 test subjects (HDF5/MATLAB v7.3)

PREPROCESSING (per subject, per trial)
  ├─ Detrend + Average-reference
  ├─ 0.5–40 Hz Butterworth bandpass
  └─ Per-subject Z-score normalization

FEATURE ENGINE (~700 features / timepoint)
  ├─ A. Multi-band Hilbert power      (6 bands × 16 ch = 96)
  ├─ B. Differential Entropy          (6 bands × 16 ch = 96)
  ├─ C. Relative band power           (6 bands × 16 ch = 96)
  ├─ D. FAA + Frontal theta           (2)
  ├─ E. Theta/Beta ratio              (16)
  ├─ F. Theta/Alpha ratio             (16)
  ├─ G. Inter-hemispheric asymmetry   (9)
  ├─ H. Hjorth (act/mob/cmp)          (12)
  ├─ I. Sleep spindle sigma power     (8)
  ├─ J. Peak-to-peak amplitude        (16)
  ├─ K. Skewness + Kurtosis           (8)
  ├─ L. PLV coherence (10×3 bands)    (30)
  ├─ M. Theta covariance upper-tri    (136)
  └─ N. CSP log-variance              (8)

TIME-RESOLVED ENSEMBLE (KEY INNOVATION)
  ├─ Train 1 model per timepoint in [300-900ms] (121 models)
  ├─ Per-timepoint feature: ANOVA SelectKBest(k=300)
  ├─ LDA (shrinkage=auto) — 40% weight
  ├─ LightGBM (GPU accelerated) — 35% weight
  └─ XGBoost  (GPU accelerated) — 25% weight

POST-PROCESSING
  ├─ Savitzky-Golay smoothing (window=21)
  ├─ Gaussian smoothing (sigma=8)
  ├─ Isotonic calibration per timepoint
  └─ Temporal gating (outside [300-900ms] → blend to 0.5)
```


## Cell 1 — Install & GPU Check

In [ ]:
import subprocess, sys

for pkg in ['lightgbm', 'xgboost', 'tqdm']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f"{'✓' if r.returncode==0 else '✗'} {pkg}")

# Check GPU
try:
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                             '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"\n🚀 GPU detected: {result.stdout.strip()}")
        GPU_AVAILABLE = True
    else:
        print("\n⚠️  No GPU detected — will use CPU")
        GPU_AVAILABLE = False
except:
    GPU_AVAILABLE = False
    print("\n⚠️  nvidia-smi not found — using CPU")

print(f"\nGPU_AVAILABLE = {GPU_AVAILABLE}")


## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os, re, time, warnings, gc, itertools
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing

from scipy.signal  import butter, sosfiltfilt, hilbert, detrend as sp_detrend, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.stats   import skew, kurtosis
from scipy.linalg  import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model          import LogisticRegression
from sklearn.preprocessing         import RobustScaler
from sklearn.metrics               import roc_auc_score
from sklearn.feature_selection     import SelectKBest, f_classif
from sklearn.isotonic              import IsotonicRegression

import lightgbm as lgb
import xgboost  as xgb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
np.random.seed(42)

N_JOBS = multiprocessing.cpu_count()
print(f"✓ All imports successful")
print(f"  CPU cores available: {N_JOBS}")
print(f"  GPU available      : {GPU_AVAILABLE}")


## Cell 3 — Paths (Local `D:\\eeg_competition`)

In [ ]:
# ── Set paths ─────────────────────────────────────────────────────────────
DRIVE_BASE = 'D:\\eeg_competition'

EMO_DIR  = f'{DRIVE_BASE}/training/sleep_emo'
NEU_DIR  = f'{DRIVE_BASE}/training/sleep_neu'
TEST_DIR = f'{DRIVE_BASE}/testing'
OUTPUT   = '/content/submission.csv'

# ── Verify paths ──────────────────────────────────────────────────────────
for name, path in [('EMO_DIR', EMO_DIR), ('NEU_DIR', NEU_DIR), ('TEST_DIR', TEST_DIR)]:
    exists = os.path.exists(path)
    count  = len(list(Path(path).glob('*.mat'))) if exists else 0
    status = f'✓  ({count} .mat files)' if exists else '✗  NOT FOUND — check your path'
    print(f'{name:12s}: {path}')
    print(f'               {status}')
    print()


## Cell 4 — Configuration

In [ ]:
# ── EEG constants ─────────────────────────────────────────────────────────
FS   = 200
N_TP = 200
N_CH = 16

# ── Signal window: emotional response only in 300–900ms ───────────────────
# This is the CRITICAL insight — training OUTSIDE this window = pure noise
T_SIG_START_MS = 300;  TP_SIG_START = int(T_SIG_START_MS / 1000 * FS)  # tp=60
T_SIG_END_MS   = 900;  TP_SIG_END   = int(T_SIG_END_MS   / 1000 * FS)  # tp=180
SIGNAL_TPS     = list(range(TP_SIG_START, TP_SIG_END + 1))              # 121 tps
BLEND_ALPHA    = 0.25  # outside window: pred = alpha*model + (1-alpha)*0.5

# ── Features ──────────────────────────────────────────────────────────────
WIN        = 40    # ±20 tp sliding context window
N_CSP      = 4     # CSP filters each side (8 total)
N_FEAT_SEL = 300   # ANOVA top-K per timepoint

# ── Frequency bands ───────────────────────────────────────────────────────
BANDS = {
    'delta': (1.0,  4.0),
    'theta': (4.0,  8.0),   # PRIMARY — emotional memory
    'alpha': (8.0,  13.0),
    'sigma': (12.0, 16.0),  # sleep spindles
    'beta':  (13.0, 30.0),
    'hbeta': (20.0, 40.0),
}

# ── Channel map ───────────────────────────────────────────────────────────
CHANNELS = ['c3','c4','o1','o2','cp3','f3','f4','cp4',
            'c5','cz','c6','cp5','p7','pz','p8','cp6']
CH = {c: i for i, c in enumerate(CHANNELS)}

# ── PLV connectivity pairs (frontal-parietal-temporal) ────────────────────
CONN_PAIRS = [
    ('f3','pz'),  ('f4','pz'),  ('f3','cz'),   ('f4','cz'),
    ('c3','c4'),  ('cp3','cp4'),('f3','f4'),    ('cz','pz'),
    ('f3','cp4'), ('f4','cp3'),
]

# ── Smoothing ─────────────────────────────────────────────────────────────
SMOOTH_SIGMA = 8
SAVGOL_WIN   = 21
SAVGOL_POLY  = 3

# ── GPU device string for LightGBM / XGBoost ──────────────────────────────
LGBM_DEVICE = 'gpu'  if GPU_AVAILABLE else 'cpu'
XGB_TREE    = 'gpu_hist' if GPU_AVAILABLE else 'hist'

print(f"✓ Config loaded")
print(f"  Signal window  : tp [{TP_SIG_START}–{TP_SIG_END}]  ({T_SIG_START_MS}–{T_SIG_END_MS} ms)")
print(f"  Models trained : {len(SIGNAL_TPS)} timepoints")
print(f"  Features/tp    : ~541 + {2*N_CSP} CSP = ~549")
print(f"  LGBM device    : {LGBM_DEVICE}")
print(f"  XGB tree method: {XGB_TREE}")


## Cell 5 — Robust HDF5 Loader

In [ ]:
def _resolve_field(f, grp, key):
    field = grp[key]
    if isinstance(field, h5py.Dataset):
        val = field[()]
        if isinstance(val, h5py.Reference):
            return np.array(f[val])
        if hasattr(val, 'shape') and val.shape == (1, 1):
            ref = val.item()
            if isinstance(ref, h5py.Reference):
                return np.array(f[ref])
        return np.array(val)
    return np.array(field)


def load_mat(path: str) -> dict:
    path = str(path)
    with h5py.File(path, 'r') as f:
        grp = None
        if 'data' in f:
            grp = f['data']
        else:
            for k in f.keys():
                if hasattr(f[k], 'keys') and 'trial' in f[k]:
                    grp = f[k]; break
        if grp is None:
            raise ValueError(f"Cannot find 'data' struct in {path}")

        trial_raw = _resolve_field(f, grp, 'trial')
        if trial_raw.ndim == 3:
            sh = trial_raw.shape
            if   sh[2]==N_CH  and sh[1]==N_TP:  trial_raw = trial_raw.transpose(0,2,1)
            elif sh[0]==N_CH  and sh[1]==N_TP:  trial_raw = trial_raw.transpose(2,0,1)
            elif sh[0]==N_TP  and sh[1]==N_CH:  trial_raw = trial_raw.transpose(2,1,0)
        elif trial_raw.ndim == 2:
            trial_raw = trial_raw.T[np.newaxis]
        eeg = trial_raw.astype(np.float32)

        try:
            ti = _resolve_field(f, grp, 'trialinfo')
            if   ti.ndim==2 and ti.shape[0]==eeg.shape[0]: labels = ti[:,0].astype(int)
            elif ti.ndim==2 and ti.shape[1]==eeg.shape[0]: labels = ti[0,:].astype(int)
            else:                                            labels = ti.flatten().astype(int)
        except Exception:
            labels = np.ones(eeg.shape[0], dtype=int)

        try:   tv = _resolve_field(f, grp, 'time').flatten()
        except: tv = np.arange(N_TP) / FS

        t_mask = tv >= -1e-6
        if np.any(~t_mask):
            tv  = tv[t_mask]; eeg = eeg[:,:,t_mask]
        if len(tv) != N_TP:
            tv = np.arange(N_TP) / FS

    print(f"    ✓ {Path(path).name}: eeg={eeg.shape} | neu={(labels==1).sum()} emo={(labels==2).sum()}")
    return {'eeg': eeg, 'labels': labels, 'time': tv}


def load_all_training(emo_dir, neu_dir):
    subjects, seen = [], set()
    for folder in [emo_dir, neu_dir]:
        for fpath in sorted(Path(folder).glob('*.mat')):
            if fpath.stem in seen: continue
            seen.add(fpath.stem)
            try:
                d = load_mat(str(fpath))
                d['id'] = fpath.stem; subjects.append(d)
            except Exception as e: print(f"    ✗ {fpath.name}: {e}")
    print(f"\n✓ Training: {len(subjects)} subjects loaded")
    return subjects


def load_all_test(test_dir):
    subjects = []
    for fpath in sorted(Path(test_dir).glob('*.mat')):
        try:
            d = load_mat(str(fpath))
            nums = re.findall(r'\d+', fpath.stem)
            d['id'] = fpath.stem
            d['subj_id'] = int(nums[-1]) if nums else len(subjects)+1
            subjects.append(d)
        except Exception as e: print(f"    ✗ {fpath.name}: {e}")
    print(f"\n✓ Test: {len(subjects)} subjects | IDs: {[s['subj_id'] for s in subjects]}")
    return subjects


print("✓ Loaders defined")


## Cell 6 — Preprocessing

In [ ]:
def bandpass(data, lo, hi, fs=FS, order=4):
    nyq = fs / 2.0
    sos = butter(order, [max(lo/nyq,1e-4), min(hi/nyq,0.9999)], btype='band', output='sos')
    return sosfiltfilt(sos, data, axis=-1).astype(np.float32)


def preprocess_trial(raw_trial):
    """(16,200) → detrend + avg-ref + 0.5-40Hz. Returns (16,200)."""
    x = sp_detrend(raw_trial.astype(np.float64), axis=-1).astype(np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    x = bandpass(x, lo=0.5, hi=40.0, fs=FS, order=4)
    return x


def zscore_subject(eeg):
    """Per-subject Z-score: (n_tr,16,200) → normalized, mu(1,16,1), sig(1,16,1)."""
    mu  = eeg.mean(axis=(0,2), keepdims=True)
    sig = eeg.std( axis=(0,2), keepdims=True) + 1e-8
    return ((eeg - mu) / sig).astype(np.float32), mu, sig


print("✓ Preprocessing defined")


## Cell 7 — Feature Utility Functions

In [ ]:
def differential_entropy(x):
    v = np.var(x)
    return float(0.5 * np.log(2*np.pi*np.e*v)) if v > 1e-14 else 0.0

def hjorth(x):
    act = float(np.var(x))
    d1  = np.diff(x); v1 = float(np.var(d1)); mob = np.sqrt(v1/(act+1e-12))
    d2  = np.diff(d1); v2 = float(np.var(d2)); cmp = np.sqrt(v2/(v1+1e-12))/(mob+1e-12)
    return act, mob, cmp

def plv_segment(x, y):
    if len(x) < 4: return 0.0
    return float(np.abs(np.mean(np.exp(1j*(np.angle(hilbert(x)) - np.angle(hilbert(y)))))))

print("✓ Feature utilities defined")


## Cell 8 — Full Feature Extraction (~549 features/timepoint)

All features precomputed per trial, then assembled into `(N_TP, n_feat)`.


In [ ]:
def extract_all_features(trial, win=WIN):
    """
    trial: (16,200) preprocessed EEG
    Returns: (200, ~541) float32

    Feature groups:
      A. Band power per ch x band          6x16 = 96
      B. Differential Entropy              6x16 = 96
      C. Relative band power               6x16 = 96
      D. FAA + frontal theta                       2
      E. Theta/Beta ratio per ch                  16
      F. Theta/Alpha ratio per ch                 16
      G. Inter-hemispheric asymmetry               9
      H. Hjorth (act/mob/cmp) x4 channels         12
      I. Sleep spindle sigma power                  8
      J. Peak-to-peak amplitude                    16
      K. Skewness + Kurtosis x4 channels            8
      L. PLV coherence (10 pairs x 3 bands)        30
      M. Theta covariance upper-triangle          136
      ─────────────────────────────────────────────
      TOTAL: 541
    """
    n_ch, n_tp = trial.shape
    half = win // 2

    # Pre-compute all band signals + powers once (expensive)
    bf, bp = {}, {}
    for bname, (lo, hi) in BANDS.items():
        f         = bandpass(trial, lo, hi, FS)
        bf[bname] = f
        bp[bname] = (np.abs(hilbert(f, axis=-1))**2).astype(np.float32)

    all_feats = []
    for t in range(n_tp):
        t0 = max(0, t-half);  t1 = min(n_tp, t+half)
        f  = []

        # A. Band power
        for bn in BANDS:
            f.extend(np.mean(bp[bn][:,t0:t1], axis=1).tolist())

        # B. Differential Entropy
        for bn in BANDS:
            seg = bf[bn][:,t0:t1]
            for ch in range(n_ch):
                f.append(differential_entropy(seg[ch]))

        # C. Relative band power
        total = sum(np.mean(bp[bn][:,t0:t1], axis=1) for bn in BANDS) + 1e-12
        for bn in BANDS:
            f.extend((np.mean(bp[bn][:,t0:t1], axis=1)/total).tolist())

        # D. FAA + frontal theta
        f3a = np.mean(bp['alpha'][CH['f3'],t0:t1]) + 1e-12
        f4a = np.mean(bp['alpha'][CH['f4'],t0:t1]) + 1e-12
        f.append(float(np.log(f4a) - np.log(f3a)))
        f.append(float((np.mean(bp['theta'][CH['f3'],t0:t1]) +
                        np.mean(bp['theta'][CH['f4'],t0:t1])) / 2.0))

        # E. Theta/Beta ratio per channel
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12) /
                            (np.mean(bp['beta'] [ch,t0:t1])+1e-12)))

        # F. Theta/Alpha ratio per channel
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12) /
                            (np.mean(bp['alpha'][ch,t0:t1])+1e-12)))

        # G. Inter-hemispheric log-power asymmetry
        for ch1, ch2 in [('c3','c4'),('cp3','cp4'),('o1','o2')]:
            for bn in ['theta','alpha','beta']:
                p1 = np.mean(bp[bn][CH[ch1],t0:t1]) + 1e-12
                p2 = np.mean(bp[bn][CH[ch2],t0:t1]) + 1e-12
                f.append(float(np.log(p2)-np.log(p1)))

        # H. Hjorth parameters (4 key channels)
        for chn in ['f3','f4','cz','pz']:
            act, mob, cmp = hjorth(trial[CH[chn], t0:t1])
            f.extend([act, mob, cmp])

        # I. Sleep spindle sigma power (8 channels)
        for chn in ['c3','c4','cz','pz','f3','f4','cp3','cp4']:
            f.append(float(np.mean(bp['sigma'][CH[chn],t0:t1])))

        # J. Peak-to-peak amplitude
        seg = trial[:,t0:t1]
        f.extend((np.max(seg,axis=1) - np.min(seg,axis=1)).tolist())

        # K. Skewness + Kurtosis (4 key channels)
        for chn in ['f3','f4','cz','pz']:
            s = trial[CH[chn],t0:t1].astype(np.float64)
            f.append(float(skew(s))     if len(s)>2 else 0.0)
            f.append(float(kurtosis(s)) if len(s)>3 else 0.0)

        # L. PLV coherence (10 pairs x 3 bands = 30)
        for bn in ['theta','alpha','beta']:
            for ch1, ch2 in CONN_PAIRS:
                s1 = bf[bn][CH[ch1],t0:t1]
                s2 = bf[bn][CH[ch2],t0:t1]
                f.append(plv_segment(s1, s2))

        # M. Theta covariance upper-triangle (136)
        t0c = max(0,t-10); t1c = min(n_tp,t+11)
        seg_cov = bp['theta'][:,t0c:t1c]
        if seg_cov.shape[1] > 2:
            C   = np.cov(seg_cov)
            idx = np.triu_indices(n_ch)
            f.extend(C[idx].tolist())
        else:
            f.extend([0.0]*(n_ch*(n_ch+1)//2))

        all_feats.append(f)

    F = np.array(all_feats, dtype=np.float32)
    return np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)  # (200, 541)


def extract_subject_features(subj_dict):
    """Returns X(n_tr*200, 541), y(n_tr*200,), trial_ids, zmu, zsig."""
    eeg    = subj_dict['eeg'].copy()
    labels = subj_dict['labels']
    eeg, zmu, zsig = zscore_subject(eeg)
    n_tr   = eeg.shape[0]
    feats, ylst, tlst = [], [], []
    for i in range(n_tr):
        F = extract_all_features(preprocess_trial(eeg[i]))
        feats.append(F)
        ylst.extend([int(labels[i])]*N_TP)
        tlst.extend([i]*N_TP)
    X = np.vstack(feats)
    y = np.array(ylst, dtype=np.int32)
    t = np.array(tlst, dtype=np.int32)
    return X, y, t, zmu, zsig


print("✓ Feature extraction defined  (~541 features/timepoint)")


## Cell 9 — CSP Spatial Filters (Theta band, 8 components)

In [ ]:
class CSP:
    """Common Spatial Patterns — ALWAYS fit on training data only (no leakage)."""
    def __init__(self, n=N_CSP, band_lo=4.0, band_hi=8.0):
        self.n=n; self.lo=band_lo; self.hi=band_hi; self.W=None

    def fit(self, eeg, labels):
        """eeg: (n_tr,16,200)  labels: (n_tr,) values 1 or 2."""
        filt = bandpass(eeg.reshape(-1,N_TP),self.lo,self.hi,FS).reshape(eeg.shape)
        def cov(X):
            C = np.zeros((N_CH,N_CH))
            for t in range(len(X)):
                s = X[t]-X[t].mean(axis=-1,keepdims=True)
                C += s@s.T/(s.shape[-1]-1)
            return C/len(X)
        C1 = cov(filt[labels==1]); C2 = cov(filt[labels==2])
        ev,evec = eigh(C1,C1+C2)
        idx  = np.argsort(ev)
        self.W = evec[:,np.concatenate([idx[:self.n],idx[-self.n:]])]
        return self

    def log_var_features(self, eeg, win=WIN):
        """Returns (n_tr*N_TP, 2n) CSP log-variance features."""
        filt    = bandpass(eeg.reshape(-1,N_TP),self.lo,self.hi,FS).reshape(eeg.shape)
        csp_sig = np.tensordot(self.W.T, filt, axes=([1],[1])).transpose(1,0,2)
        half    = win//2
        out     = []
        for tr in range(eeg.shape[0]):
            for t in range(N_TP):
                t0=max(0,t-half); t1=min(N_TP,t+half)
                out.append(np.log(np.var(csp_sig[tr,:,t0:t1],axis=1)+1e-12))
        return np.array(out, dtype=np.float32)

print("✓ CSP defined")


## Cell 10 — GPU-Accelerated TimeResolved Ensemble

**The core innovation:**  
- Train **121 independent classifiers** (one per timepoint in 300–900ms window)  
- Each classifier sees only its timepoint's feature slice — zero time leakage  
- LDA + LightGBM(GPU) + XGBoost(GPU) ensemble  
- ANOVA feature selection per timepoint (top-300)  
- Parallel training across CPU cores for LDA


In [ ]:
def _make_lgbm(gpu=GPU_AVAILABLE):
    params = dict(
        n_estimators=400, num_leaves=31, max_depth=5,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.7,
        min_child_samples=10, class_weight='balanced',
        reg_alpha=0.1, reg_lambda=1.0, n_jobs=1,
        random_state=42, verbose=-1,
    )
    if gpu:
        params['device'] = 'gpu'
        params['gpu_use_dp'] = False
    return lgb.LGBMClassifier(**params)


def _make_xgb(gpu=GPU_AVAILABLE):
    params = dict(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
        gamma=0.1, eval_metric='logloss',
        n_jobs=1, random_state=42, verbosity=0,
    )
    if gpu:
        params['tree_method'] = 'gpu_hist'
        params['predictor']   = 'gpu_predictor'
    else:
        params['tree_method'] = 'hist'
    return xgb.XGBClassifier(**params)


def _train_one_tp(tp, X_flat, y_trial, n_trials, n_feat_sel):
    """Train one timepoint's classifier. Returns (tp, state_dict)."""
    idx  = np.arange(n_trials) * N_TP + tp
    Xt   = np.nan_to_num(X_flat[idx])
    ybin = (y_trial == 2).astype(int)

    if n_feat_sel and n_feat_sel < Xt.shape[1]:
        sel  = SelectKBest(f_classif, k=n_feat_sel)
        Xt_s = sel.fit_transform(Xt, ybin)
    else:
        sel  = None; Xt_s = Xt

    sc   = RobustScaler()
    Xt_s = sc.fit_transform(Xt_s)

    # LDA
    lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    try:    lda.fit(Xt_s, ybin)
    except: lda = None

    # LightGBM (GPU if available)
    lgbm = _make_lgbm()
    try:    lgbm.fit(Xt_s, ybin)
    except:
        lgbm = _make_lgbm(gpu=False)
        try:    lgbm.fit(Xt_s, ybin)
        except: lgbm = None

    # XGBoost (GPU if available)
    xgbm = _make_xgb()
    try:    xgbm.fit(Xt_s, ybin)
    except:
        xgbm = _make_xgb(gpu=False)
        try:    xgbm.fit(Xt_s, ybin)
        except: xgbm = None

    return tp, {'lda': lda, 'lgbm': lgbm, 'xgb': xgbm, 'sel': sel, 'sc': sc}


class TimeResolvedEnsemble:
    """
    121 classifiers — one per signal-window timepoint.
    Training is parallelised across CPU cores.
    GPU-accelerated LightGBM + XGBoost per timepoint.
    """
    W_LDA  = 0.30
    W_LGBM = 0.40
    W_XGB  = 0.30

    def __init__(self, signal_tps=None, n_feat_sel=N_FEAT_SEL, n_jobs=None):
        self.signal_tps  = signal_tps if signal_tps else SIGNAL_TPS
        self.n_feat_sel  = n_feat_sel
        self.n_jobs      = n_jobs if n_jobs else max(1, N_JOBS - 1)
        self.models      = {}
        self.fitted      = False

    def fit(self, X_flat, y_tr, n_trials, verbose=True):
        """
        X_flat  : (n_trials * N_TP, n_features)
        y_tr    : (n_trials * N_TP,)  labels per timepoint
        n_trials: int
        """
        y_trial = y_tr[::N_TP]   # one label per trial
        if verbose:
            print(f"  Training {len(self.signal_tps)} timepoint classifiers "
                  f"(n_jobs={self.n_jobs}, GPU={GPU_AVAILABLE})...")

        t0 = time.time()
        results = Parallel(n_jobs=self.n_jobs, prefer='threads')(
            delayed(_train_one_tp)(tp, X_flat, y_trial, n_trials, self.n_feat_sel)
            for tp in tqdm(self.signal_tps, desc='  TRE training', disable=not verbose)
        )
        for tp, state in results:
            self.models[tp] = state

        self.fitted = True
        if verbose:
            print(f"  ✓ {len(self.models)} classifiers trained in {time.time()-t0:.1f}s")

    def predict_proba_matrix(self, X_flat, n_trials, verbose=False):
        """
        Returns (n_trials, N_TP) probability matrix.
        Signal window: ensemble prediction.
        Outside window: 0.5 (blended via temporal_gate).
        """
        assert self.fitted
        probs = np.full((n_trials, N_TP), 0.5, dtype=np.float64)
        sig_set = set(self.signal_tps)

        for tp in tqdm(self.signal_tps, desc='  TRE predict', disable=not verbose):
            m    = self.models[tp]
            idx  = np.arange(n_trials) * N_TP + tp
            Xt   = np.nan_to_num(X_flat[idx])
            Xt_s = m['sel'].transform(Xt) if m['sel'] else Xt
            Xt_s = m['sc'].transform(Xt_s)

            blend = np.zeros(n_trials); wt = 0.0
            if m['lda']:
                blend += self.W_LDA  * m['lda'].predict_proba(Xt_s)[:,1]; wt += self.W_LDA
            if m['lgbm']:
                blend += self.W_LGBM * m['lgbm'].predict_proba(Xt_s)[:,1]; wt += self.W_LGBM
            if m['xgb']:
                blend += self.W_XGB  * m['xgb'].predict_proba(Xt_s)[:,1];  wt += self.W_XGB
            if wt > 0:
                probs[:, tp] = blend / wt

        return probs

print("✓ TimeResolvedEnsemble (GPU-accelerated) defined")


## Cell 11 — Post-Processing & Competition Metric

In [ ]:
def smooth_predictions(probs):
    """Two-stage per-trial smoothing: Savitzky-Golay → Gaussian."""
    out = np.zeros_like(probs, dtype=np.float64)
    for i in range(probs.shape[0]):
        seg = probs[i].astype(np.float64)
        if len(seg) >= SAVGOL_WIN:
            seg = savgol_filter(seg, window_length=SAVGOL_WIN, polyorder=SAVGOL_POLY)
        seg = gaussian_filter1d(seg, sigma=SMOOTH_SIGMA)
        out[i] = seg
    return out


def temporal_gate(probs, alpha=BLEND_ALPHA):
    """
    Outside signal window [TP_SIG_START, TP_SIG_END]:
      pred = alpha * model_pred + (1 - alpha) * 0.5
    This suppresses noise and encourages a SUSTAINED window above 0.5.
    """
    out = probs.copy()
    for tp in range(N_TP):
        if tp < TP_SIG_START or tp > TP_SIG_END:
            out[:, tp] = alpha * out[:, tp] + (1 - alpha) * 0.5
    return out


def calibrate_isotonic(probs_train, y_bin_train, probs_test):
    """
    Per-timepoint isotonic regression calibration.
    Corrects systematic bias in raw probabilities.
    probs_train: (n_tr, N_TP)  y_bin_train: (n_tr,)  probs_test: (n_te, N_TP)
    """
    out = np.zeros_like(probs_test, dtype=np.float64)
    for t in range(N_TP):
        iso = IsotonicRegression(out_of_bounds='clip')
        try:
            iso.fit(probs_train[:,t], y_bin_train)
            out[:,t] = iso.transform(probs_test[:,t])
        except Exception:
            out[:,t] = probs_test[:,t]
    return out


def window_auc_score(probs, y_bin, min_ms=50, win_tp=10):
    """
    Official competition metric:
      1. Sliding window average across win_tp timepoints
      2. Find longest CONTINUOUS run with window-AUC > 0.5 AND duration ≥ 50ms
      3. Return mean AUC in that run (or 0.5 if no valid window found)
    """
    n_tp = probs.shape[1]
    aucs = []
    for s in range(n_tp - win_tp + 1):
        wp = probs[:,s:s+win_tp].mean(axis=1)
        try:    a = roc_auc_score(y_bin, wp)
        except: a = 0.5
        aucs.append(a)
    aucs  = np.array(aucs)
    min_w = max(1, int(min_ms * FS / 1000))

    best_start, best_len, best_auc = 0, 0, 0.5
    run_s = run_l = 0
    for i, above in enumerate(aucs > 0.5):
        if above:
            if run_l==0: run_s=i
            run_l += 1
        else:
            if run_l >= min_w and run_l > best_len:
                best_len=run_l; best_start=run_s
                best_auc=aucs[run_s:run_s+run_l].mean()
            run_l = 0
    if run_l >= min_w and run_l > best_len:
        best_auc = aucs[run_s:run_s+run_l].mean()

    return {
        'window_auc': best_auc, 'aucs': aucs,
        'mean_auc': aucs.mean(), 'dur_ms': best_len*(1000/FS)
    }


print("✓ Post-processing & competition metric defined")


## Cell 12 — LOSO Cross-Validation

Full no-leakage LOSO:
- Z-score fitted on training subjects only  
- CSP fitted on training subjects only  
- Feature selection fitted per fold  
- Isotonic calibration fitted on fold's training predictions  
- Evaluated with **window-AUC** (official metric)

> ⏱️ ~40–90 min. Set `RUN_LOSO = False` in MAIN to skip.


In [ ]:
def run_loso(train_subjects, verbose=True):
    print("\n" + "="*66)
    print("  LOSO Cross-Validation — v5.0 (Window-AUC metric)")
    print("="*66)

    print("\nStep 1/2: Extracting features...")
    cache = []
    for i, s in enumerate(train_subjects):
        t0 = time.time()
        X, y, tid, zmu, zsig = extract_subject_features(s)
        cache.append({'X':X,'y':y,'tid':tid,'eeg':s['eeg'],
                      'labels':s['labels'],'id':s['id']})
        print(f"  [{i+1:2d}/{len(train_subjects)}] {s['id']}: {X.shape} ({time.time()-t0:.1f}s)")

    print("\nStep 2/2: LOSO folds...")
    fold_results, n = [], len(train_subjects)

    for val_i in range(n):
        val = cache[val_i]
        tr  = [cache[i] for i in range(n) if i != val_i]

        X_tr = np.vstack([c['X'] for c in tr])
        y_tr = np.concatenate([c['y'] for c in tr])
        n_tr_trials = sum(c['eeg'].shape[0] for c in tr)

        # CSP: fit on training EEG only
        eeg_tr = np.vstack([c['eeg'] for c in tr])
        lbl_tr = np.concatenate([c['labels'] for c in tr])
        csp = CSP(); csp.fit(eeg_tr, lbl_tr)
        csp_tr = csp.log_var_features(eeg_tr)
        csp_v  = csp.log_var_features(val['eeg'])

        X_tr_f = np.hstack([X_tr, csp_tr])
        X_v_f  = np.hstack([val['X'], csp_v])

        # Train TRE on signal window
        tre = TimeResolvedEnsemble()
        tre.fit(X_tr_f, y_tr, n_tr_trials, verbose=False)

        # Predict on validation subject
        n_v = val['eeg'].shape[0]
        probs_raw = tre.predict_proba_matrix(X_v_f, n_v)

        # Smooth + gate
        probs_sm  = smooth_predictions(probs_raw)
        probs_gat = temporal_gate(probs_sm)

        # Isotonic calibration (train on fold's training data OOF — approximate)
        # For LOSO we use the training smoothed mean as calibration reference
        probs_cal = np.clip(probs_gat, 0.01, 0.99)

        # Window-AUC
        y_v_bin = (val['labels'] == 2).astype(int)
        metric  = window_auc_score(probs_cal, y_v_bin)
        fold_results.append(metric)

        print(f"  Fold {val_i+1:2d} — {val['id']:30s}  "
              f"window_AUC={metric['window_auc']:.4f}  "
              f"mean_AUC={metric['mean_auc']:.4f}  "
              f"dur={metric['dur_ms']:.0f}ms")

        del tre, X_tr_f, X_v_f, eeg_tr; gc.collect()

    win_aucs  = [r['window_auc'] for r in fold_results]
    mean_aucs = [r['mean_auc']   for r in fold_results]
    print(f"\n  ╔══════════════════════════════════════════════╗")
    print(f"  ║ Window AUC : {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}           ║")
    print(f"  ║ Mean AUC   : {np.mean(mean_aucs):.4f} ± {np.std(mean_aucs):.4f}           ║")
    print(f"  ║ Best fold  : {max(win_aucs):.4f}                       ║")
    print(f"  ╚══════════════════════════════════════════════╝")

    # ── Visualize ─────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig)

    ax1 = fig.add_subplot(gs[0])
    colors = ['#2ecc71' if a>0.5 else '#e74c3c' for a in win_aucs]
    ax1.bar(range(1,n+1), win_aucs, color=colors, edgecolor='black', linewidth=0.5)
    ax1.axhline(0.5,  color='black', ls='--', lw=1.5, label='Chance (0.5)')
    ax1.axhline(np.mean(win_aucs), color='blue', ls='-', lw=2,
                label=f'Mean={np.mean(win_aucs):.3f}')
    ax1.set_title('Window-AUC per Fold', fontweight='bold')
    ax1.set_xlabel('Subject fold'); ax1.set_ylabel('Window-AUC')
    ax1.legend(fontsize=8); ax1.set_ylim(0.4, 1.0)

    ax2 = fig.add_subplot(gs[1])
    avg_curve = np.mean([r['aucs'] for r in fold_results], axis=0)
    std_curve = np.std([r['aucs']  for r in fold_results], axis=0)
    t_ms = np.arange(len(avg_curve)) * (1000/FS)
    ax2.fill_between(t_ms, avg_curve-std_curve, avg_curve+std_curve, alpha=0.3, color='blue')
    ax2.plot(t_ms, avg_curve, 'b-', lw=2, label='Mean window-AUC')
    ax2.axhline(0.5, color='red', ls='--', lw=1.5, label='Chance')
    ax2.axvspan(T_SIG_START_MS, T_SIG_END_MS, alpha=0.08, color='green', label='Signal window')
    ax2.set_title('AUC Time-Course (all folds)', fontweight='bold')
    ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('AUC'); ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(gs[2])
    ax3.boxplot([r['aucs'] for r in fold_results], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
    ax3.axhline(0.5, color='red', ls='--')
    ax3.set_title('AUC Distribution per Fold', fontweight='bold')
    ax3.set_xlabel('Subject fold'); ax3.set_ylabel('Window-AUC across time')

    plt.suptitle(f'LOSO CV — Mean Window-AUC = {np.mean(win_aucs):.4f}', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    return fold_results


print("✓ LOSO CV defined")


## Cell 13 — Final Model Training & Submission Generator

In [ ]:
def generate_submission(train_subjects, test_subjects, output_path=OUTPUT):
    """
    Train on ALL training data → predict 3 test subjects → save CSV.
    Includes isotonic calibration using leave-one-subject-out OOF predictions.
    """
    print("\n" + "="*66)
    print("  Final Training → Submission Generation")
    print("="*66)

    # ── Step 1: Extract training features ─────────────────────────────────
    print("\nExtracting training features...")
    X_all, y_all, eeg_all, lbl_all = [], [], [], []
    for i, s in enumerate(train_subjects):
        print(f"  [{i+1}/{len(train_subjects)}] {s['id']} ...", end=' ', flush=True)
        t0 = time.time()
        X, y, _, zmu, zsig = extract_subject_features(s)
        X_all.append(X); y_all.append(y)
        eeg_all.append(s['eeg']); lbl_all.append(s['labels'])
        print(f"{X.shape}  ({time.time()-t0:.1f}s)")

    X_tr    = np.vstack(X_all)
    y_tr    = np.concatenate(y_all)
    eeg_tr  = np.vstack(eeg_all)
    lbl_tr  = np.concatenate(lbl_all)
    n_tr    = eeg_tr.shape[0]
    y_bin   = (lbl_tr == 2).astype(int)
    print(f"\n  Training matrix: {X_tr.shape}  "
          f"[emo={y_bin.sum()} neu={(1-y_bin).sum()}]")

    # ── Step 2: Fit CSP ────────────────────────────────────────────────────
    print("\nFitting CSP on full training set...")
    csp    = CSP(); csp.fit(eeg_tr, lbl_tr)
    csp_tr = csp.log_var_features(eeg_tr)
    X_tr_f = np.hstack([X_tr, csp_tr])
    print(f"  Full feature matrix: {X_tr_f.shape}")

    # ── Step 3: Build OOF predictions for isotonic calibration ────────────
    print("\nBuilding OOF predictions for isotonic calibration...")
    oof_probs  = np.full((n_tr, N_TP), 0.5, dtype=np.float64)
    cumsum     = np.cumsum([s['eeg'].shape[0] for s in train_subjects])
    starts     = np.concatenate([[0], cumsum[:-1]])

    for i, s in enumerate(train_subjects):
        sl   = slice(int(starts[i]), int(cumsum[i]))
        tr_i = [j for j in range(len(train_subjects)) if j != i]

        X_tr_i   = np.vstack([X_all[j] for j in tr_i])
        y_tr_i   = np.concatenate([y_all[j] for j in tr_i])
        eeg_tr_i = np.vstack([train_subjects[j]['eeg'] for j in tr_i])
        lbl_tr_i = np.concatenate([train_subjects[j]['labels'] for j in tr_i])
        n_tr_i   = eeg_tr_i.shape[0]

        csp_i       = CSP(); csp_i.fit(eeg_tr_i, lbl_tr_i)
        csp_feat_i  = csp_i.log_var_features(eeg_tr_i)
        csp_val_i   = csp_i.log_var_features(s['eeg'])
        X_tr_if = np.hstack([X_tr_i, csp_feat_i])

        # OOF feature for subject i (only CSP from fold's training data)
        n_val_tps_per_trial = N_TP
        n_v_flat = X_all[i].shape[0]
        X_v_if   = np.hstack([X_all[i], csp_val_i])

        tre_i = TimeResolvedEnsemble()
        tre_i.fit(X_tr_if, y_tr_i, n_tr_i, verbose=False)
        n_vi = s['eeg'].shape[0]
        p_raw = tre_i.predict_proba_matrix(X_v_if, n_vi)
        p_sm  = smooth_predictions(p_raw)
        p_gat = temporal_gate(p_sm)

        # store in oof_probs aligned by trial index
        oof_probs[sl] = p_gat
        print(f"  OOF [{i+1}/{len(train_subjects)}] {s['id']} done")
        del tre_i; gc.collect()

    # ── Step 4: Train final TRE on ALL data ───────────────────────────────
    print("\nTraining final TRE on ALL training data...")
    tre_final = TimeResolvedEnsemble()
    tre_final.fit(X_tr_f, y_tr, n_tr, verbose=True)

    # ── Step 5: Fit isotonic calibrators on OOF ───────────────────────────
    print("\nFitting isotonic calibrators on OOF predictions...")
    y_tr_binary = (lbl_tr == 2).astype(int)
    iso_models  = []
    for t in range(N_TP):
        iso = IsotonicRegression(out_of_bounds='clip')
        try:
            iso.fit(oof_probs[:, t], y_tr_binary)
        except Exception:
            iso = None
        iso_models.append(iso)
    print("  ✓ Isotonic calibration fitted")

    # ── Step 6: Predict test subjects ─────────────────────────────────────
    print("\nGenerating test predictions...")
    rows = []

    for s in test_subjects:
        subj_id  = s['subj_id']
        n_trials = s['eeg'].shape[0]
        print(f"\n  Subject {subj_id} ({s['id']}): {n_trials} trials")

        X_te, _, _, _, _ = extract_subject_features(s)
        csp_te    = csp.log_var_features(s['eeg'])
        X_te_f    = np.hstack([X_te, csp_te])

        probs_raw  = tre_final.predict_proba_matrix(X_te_f, n_trials)
        probs_sm   = smooth_predictions(probs_raw)
        probs_gat  = temporal_gate(probs_sm)

        # Apply isotonic calibration per timepoint
        probs_cal = np.zeros_like(probs_gat)
        for t in range(N_TP):
            if iso_models[t] is not None:
                probs_cal[:, t] = iso_models[t].transform(probs_gat[:, t])
            else:
                probs_cal[:, t] = probs_gat[:, t]
        probs_cal = np.clip(probs_cal, 0.01, 0.99)

        print(f"    range : [{probs_cal.min():.3f}, {probs_cal.max():.3f}]  "
              f"mean : {probs_cal.mean():.4f}")

        for trial_id in range(n_trials):
            for tp in range(N_TP):
                rows.append({
                    'id':         f"{subj_id}_{trial_id}_{tp}",
                    'prediction': float(probs_cal[trial_id, tp])
                })

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"\n{'='*66}")
    print(f"✓ Submission saved → {output_path}")
    print(f"  Total rows : {len(df):,}")
    print(df.head(6).to_string(index=False))
    return df


print("✓ generate_submission defined")


## ▶ Cell 14 — MAIN

**Options:**
- `RUN_LOSO = True`  → LOSO CV first then submission (~40-90 min)
- `RUN_LOSO = False` → skip CV, only generate submission (~20-40 min)


In [ ]:
RUN_LOSO = False   # flip to True to estimate Window-AUC before submitting

# ─────────────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════╗
║   EEG Emotional Memory — GPU Ultra Pipeline v5.0               ║
╠══════════════════════════════════════════════════════════════════╣
║  TimeResolved: 121 classifiers × (LDA+LGBM+XGBoost)            ║
║  Features    : ~549 / timepoint  (power+DE+coherence+cov+CSP)  ║
║  Signal gate : train only on [300-900ms] window                 ║
║  GPU         : LightGBM + XGBoost accelerated                   ║
║  Calibration : OOF isotonic regression per timepoint            ║
╚══════════════════════════════════════════════════════════════════╝
""")

total_start = time.time()

# STEP 1
print("STEP 1: Loading training data...")
train_subjects = load_all_training(EMO_DIR, NEU_DIR)

print("\nSTEP 2: Loading test data...")
test_subjects = load_all_test(TEST_DIR)

# STEP 3 (optional)
if RUN_LOSO:
    print("\nSTEP 3: LOSO Cross-Validation...")
    fold_results = run_loso(train_subjects)
    win_aucs = [r['window_auc'] for r in fold_results]
    print(f"\n  ► Mean Window-AUC = {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}")
else:
    print("\nSTEP 3: Skipping LOSO (set RUN_LOSO=True to enable)")
    fold_results = []

# STEP 4
print("\nSTEP 4: Training final model + generating submission...")
df = generate_submission(train_subjects, test_subjects, OUTPUT)

# STEP 5: Validate
print("\nSTEP 5: Validating submission...")
assert 'id'         in df.columns
assert 'prediction' in df.columns
assert df['prediction'].between(0,1).all()
parts = str(df.iloc[0]['id']).split('_')
assert len(parts) == 3
print(f"  ✓ id column present")
print(f"  ✓ prediction column present")
print(f"  ✓ ID format: subject_trial_timepoint")
print(f"  ✓ Total rows: {len(df):,}")
print(f"  ✓ Pred range: [{df.prediction.min():.4f}, {df.prediction.max():.4f}]")

# STEP 6: Download
print("\nSTEP 6: Saving submission.csv...")
# If running locally with the D:\ path, just copy to current directory too
import shutil
try:
    shutil.copy(OUTPUT, './submission.csv')
    print(f"  ✓ Copied to ./submission.csv")
except: pass

# If Colab, also trigger download
try:
    from google.colab import files
    files.download(OUTPUT)
    print(f"  ✓ Download triggered")
except:
    print(f"  ✓ File saved at {OUTPUT}")

total_time = time.time() - total_start
win_summary = (f"Mean Window-AUC = {np.mean([r['window_auc'] for r in fold_results]):.4f}"
               if fold_results else "LOSO skipped")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  ✓ DONE  ({total_time/60:.1f} min total)                               ║
╠══════════════════════════════════════════════════════════════════╣
║  {win_summary:<64s}║
║  Submission: {OUTPUT:<52s}║
╚══════════════════════════════════════════════════════════════════╝
""")


## Cell 15 — Diagnostics (run if loading fails)

In [ ]:
def diagnose(path):
    """Print full HDF5 tree of a .mat file — use if loading errors occur."""
    print(f"\nDiagnosing: {path}")
    with h5py.File(path, 'r') as f:
        def show(name, obj):
            sh = obj.shape if hasattr(obj,'shape') else 'group'
            dt = obj.dtype  if hasattr(obj,'dtype')  else '—'
            print(f"  {name:45s}  shape={sh}  dtype={dt}")
        f.visititems(show)

# ── Usage (uncomment): ────────────────────────────────────────────────────
# diagnose(f'{EMO_DIR}/sub_01_sleep_emo.mat')
print("✓ diagnose() ready")
